In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 前几章的 Cell 0 都在检查 GPU，因为要加载大模型做推理。
# 本章只研究 tokenizer——加载 tokenizer 文件、训练小 tokenizer，
# 全程在 CPU 上跑，几十秒完成。所以这里不强制 GPU，只打印环境信息。
import sys
import platform

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("本章无需 GPU，CPU 运行时即可 Run All。")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，作用是把 pip 的安装日志藏起来
# transformers>=4.51:  加载 Qwen3 tokenizer 需要（Qwen3 系列的版本下界）
# tokenizers:          Hugging Face 的底层分词库，第 6.4 节用它训练 byte-level BPE
#                      （transformers 会把它作为依赖装上，这里显式写出来更清楚）
!pip install -q -U "transformers>=4.51" tokenizers

In [ ]:
# ============================================================
# Cell 2: 加载 Qwen3 tokenizer，观察它怎么切不同文本
# ============================================================
# AutoTokenizer 只读 tokenizer.json / tokenizer_config.json 等几 MB 的小文件，
# 不碰模型权重，所以 CPU、几秒就好
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
print("词表大小 vocab_size =", tokenizer.vocab_size)

# 拿四类文本各切一段：英文 / 中文 / 代码 / emoji+生僻
samples = [
    "Tokenization is fun!",
    "机器学习很有趣！",
    "x = [i**2 for i in range(10)]",
    "I love 🤖 and 北京烤鸭!",
]

for text in samples:
    # add_special_tokens=False：只切正文，不在两端加 BOS/EOS 之类，看得更干净
    ids = tokenizer.encode(text, add_special_tokens=False)
    # 把每个 id 单独 decode 回来，看清每个 token 对应的那段文本
    pieces = [tokenizer.decode([i]) for i in ids]
    print(f"\n原文: {text!r}")
    print(f"  token 数: {len(ids)}")
    print(f"  ids:    {ids}")
    print(f"  逐 token: {pieces}")

In [ ]:
# ============================================================
# Cell 3: 纯 Python 手写 BPE 训练（对应 2.1 节的 hug/pug 例子）
# ============================================================
from collections import Counter

# 语料：词 -> 频次（和文档 2.1 节完全一致）
corpus = {"hug": 10, "pug": 5, "pun": 12, "bun": 4, "hugs": 5}

# 初始：每个词按字符拆成符号列表
splits = {word: list(word) for word in corpus}

# 基础词表 = 出现过的所有字符
vocab = sorted({ch for word in corpus for ch in word})
print("基础词表 (%d):" % len(vocab), vocab)


def get_pair_freqs(splits, corpus):
    """统计所有相邻符号对的频次（按词频加权）"""
    pair_freqs = Counter()
    for word, freq in corpus.items():
        symbols = splits[word]
        for i in range(len(symbols) - 1):
            pair_freqs[(symbols[i], symbols[i + 1])] += freq
    return pair_freqs


def merge_pair(a, b, splits):
    """把所有词里相邻的 (a, b) 合并成新符号 a+b"""
    for word, symbols in splits.items():
        new_symbols, i = [], 0
        while i < len(symbols):
            # 命中相邻的 (a, b) 就合并，跳过两个；否则原样保留一个
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(a + b)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        splits[word] = new_symbols
    return splits


merges = []          # 有序的合并规则
num_merges = 3       # 学 3 轮（演示用；真实 BPE 是几万轮）
for step in range(num_merges):
    pair_freqs = get_pair_freqs(splits, corpus)
    # 选频次最高的相邻对（频次相同则按字典序，保证可复现）
    best_pair = max(pair_freqs, key=lambda p: (pair_freqs[p], p))
    splits = merge_pair(*best_pair, splits)
    merges.append(best_pair)
    vocab.append(best_pair[0] + best_pair[1])
    print(f"\nmerge {step + 1}: {best_pair} 频次={pair_freqs[best_pair]} "
          f"→ 新符号 '{best_pair[0] + best_pair[1]}'")
    print("  当前切分:", dict(splits))

print("\n最终词表 (%d):" % len(vocab), vocab)
print("合并规则（有序）:", merges)

In [ ]:
# ============================================================
# Cell 4: 用学到的合并规则编码新词（对应 2.2 节）
# ============================================================
def encode_word(word, merges):
    """把一个词拆成字符，再按 merges 的顺序逐条套用合并规则"""
    symbols = list(word)
    for a, b in merges:                     # 关键：按学习顺序，一条一条来
        new_symbols, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new_symbols.append(a + b)
                i += 2
            else:
                new_symbols.append(symbols[i])
                i += 1
        symbols = new_symbols
    return symbols


for w in ["hug", "hugs", "bug"]:
    print(f"{w!r:8} -> {encode_word(w, merges)}")

In [ ]:
# ============================================================
# Cell 5: 准备一小段训练语料（写到文件）
# ============================================================
# 真实 tokenizer 在几百 GB 语料上训；这里用一小段中英文混合文本做演示，
# 目的是看清「byte-level BPE 训出来长什么样」，而非追求质量。
corpus_text = """
Tokenization is the first step of every language model.
A tokenizer turns text into a sequence of integer ids.
Byte pair encoding merges the most frequent pair of symbols, again and again.
Large language models such as GPT, Llama and Qwen all use byte-level BPE.
The model never sees characters, only token ids.
分词是大模型的第一步，它把文本变成一串整数 id。
字节对编码反复合并出现最频繁的相邻符号对。
GPT、Llama、Qwen 等大模型都用字节级 BPE，从根上消灭未登录词。
机器学习很有趣，自然语言处理更有趣。
""" * 50  # 重复几十遍，让高频片段有足够频次被合并

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(corpus_text)

print("语料字符数:", len(corpus_text))
print("前 120 字:", corpus_text.strip()[:120])

In [ ]:
# ============================================================
# Cell 6: 配置并训练一个 byte-level BPE tokenizer
# ============================================================
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# 1) 选 BPE 作为分词模型
tok = Tokenizer(models.BPE())

# 2) pre-tokenizer：ByteLevel 会先把文本按 UTF-8 转成字节再处理
#    add_prefix_space=False：不在句首额外加空格（与 Qwen/GPT 习惯一致）
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# 3) decoder：解码时把字节正确拼回 UTF-8 文本（和 pre-tokenizer 配套）
tok.decoder = decoders.ByteLevel()

# 4) 训练器：
#    vocab_size=500   学到 500 个 token 就停（演示用，真实是几万～十几万）
#    special_tokens   手工塞进词表的特殊 token，绝不会被 BPE 拆开
#    initial_alphabet 把全部 256 个字节作为基础词表 —— byte-level「永不 OOV」的根
trainer = trainers.BpeTrainer(
    vocab_size=500,
    special_tokens=["<|endoftext|>", "<|im_start|>", "<|im_end|>"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
    show_progress=False,
)

tok.train(["corpus.txt"], trainer)

print("训练完成，词表大小:", tok.get_vocab_size())
print("基础字节数（initial_alphabet）:", len(pre_tokenizers.ByteLevel.alphabet()))
print("特殊 token 的 id:")
for s in ["<|endoftext|>", "<|im_start|>", "<|im_end|>"]:
    print(f"  {s} -> {tok.token_to_id(s)}")

In [ ]:
# ============================================================
# Cell 7: 用自训的 tokenizer 编码 / 解码，验证永不 OOV
# ============================================================
for text in ["language model", "机器学习", "emoji 🤖 test", "𠮷野家"]:
    enc = tok.encode(text)
    decoded = tok.decode(enc.ids)
    print(f"\n原文: {text!r}")
    print(f"  tokens: {enc.tokens}")        # byte-level token 形态，含 Ġ / 乱码字节符号
    print(f"  ids:    {enc.ids}")
    print(f"  decode: {decoded!r}")
    print(f"  完美还原: {decoded == text}")

In [ ]:
# ============================================================
# Cell 8: 中英文 token 效率对比（用 Qwen3 tokenizer）
# ============================================================
# 一段意思基本对应的中英文，看各自被切成多少 token
pairs = [
    ("Large language models are powerful.", "大语言模型很强大。"),
    ("Machine learning is a subfield of artificial intelligence.",
     "机器学习是人工智能的一个子领域。"),
]

for en, zh in pairs:
    en_ids = tokenizer.encode(en, add_special_tokens=False)
    zh_ids = tokenizer.encode(zh, add_special_tokens=False)
    print(f"\nEN: {en!r}")
    print(f"   {len(en)} 字符 -> {len(en_ids)} token  "
          f"(每 token 约 {len(en) / len(en_ids):.2f} 字符)")
    print(f"ZH: {zh!r}")
    print(f"   {len(zh)} 字符 -> {len(zh_ids)} token  "
          f"(每 token 约 {len(zh) / len(zh_ids):.2f} 字符)")